# F08: File-Drop Ingestion Testing (Fabric Scenario)

## What You'll Learn

A common Fabric ingestion pattern is the **file-drop pipeline**: upstream systems
deposit files into a landing zone (OneLake, ADLS, or a Lakehouse `Files/` folder)
on a daily cadence, and a Data Pipeline or Spark notebook picks them up, validates
them, and loads them into tables.

Testing this pattern requires **realistic file drops** — not just data, but the
full operational envelope: correct file names, manifests, done-flags, late arrivals,
and the occasional corrupted file.

In this notebook you will:

1. **Understand** the file-drop ingestion pattern and its testing challenges
2. **Generate** a healthcare dataset as the source data
3. **Configure** a 14-day daily file-drop simulation
4. **Simulate** the drops with late arrivals
5. **Inject** chaos into selected files
6. **Validate** with validation gates
7. **Review** the complete pipeline test structure

## The File-Drop Pattern

```
landing/
  healthcare/
    2024-01-01/
      patient.parquet
      encounter.parquet
      _manifest.json
      _done
    2024-01-02/
      ...
```

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with Fabric Data Pipelines (helpful but not required)

## Time Estimate

**~15 minutes**

In [ ]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Step 1 — Generate Healthcare Source Data

**What we're about to do:** Generate a healthcare dataset that will serve as
the source for our file-drop simulation. This represents the data that an
upstream EHR system would produce daily.

**Why this matters:** The file-drop simulator needs a source dataset to
partition into daily slices. By using the healthcare domain, we get realistic
patient, encounter, and claim data — exactly what a hospital's data feed
would contain.

**What to expect:** A standard healthcare dataset at small scale.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.healthcare import HealthcareDomain

result = Spindle().generate(domain=HealthcareDomain(), scale="small", seed=42)

print("Healthcare Source Data")
print("=" * 45)
for table_name, df in result.tables.items():
    print(f"  {table_name:<20} {len(df):>8,} rows")

total = sum(len(df) for df in result.tables.values())
print(f"\nTotal rows: {total:,}")
print(f"\nThis data will be sliced into 14 daily file drops.")

## Step 2 — Configure the File-Drop Simulation

**What we're about to do:** Set up a `FileDropConfig` that defines every aspect
of the simulation — cadence, date range, output format, lateness behavior,
manifest files, and done-flags.

**Why this matters:** Each configuration parameter maps to a real operational
concern:
- **`cadence`**: How often files land (daily, hourly, etc.)
- **`lateness_enabled` / `lateness_probability`**: Simulates upstream delays
  that your pipeline must handle gracefully
- **`manifest_enabled`**: Produces a `_manifest.json` listing expected files
  and row counts for validation
- **`done_flag_enabled`**: Creates a `_done` sentinel file that signals
  "all files for this batch have landed"

**What to expect:** A config object ready to drive the simulation.

In [ ]:
from sqllocks_spindle.simulation.file_drop import FileDropSimulator, FileDropConfig

config = FileDropConfig(
    domain="healthcare",
    base_path="./f08_demo",
    cadence="daily",
    date_range_start="2024-01-01",
    date_range_end="2024-01-14",
    formats=["parquet"],
    lateness_enabled=True,
    lateness_probability=0.15,     # 15% of drops will be late
    manifest_enabled=True,          # produce _manifest.json per batch
    done_flag_enabled=True,         # produce _done sentinel file
    seed=42,
)

print("File-Drop Configuration")
print("=" * 45)
print(f"  Domain:        {config.domain}")
print(f"  Base path:     {config.base_path}")
print(f"  Cadence:       {config.cadence}")
print(f"  Date range:    {config.date_range_start} to {config.date_range_end}")
print(f"  Format:        {config.formats}")
print(f"  Late arrivals: {config.lateness_enabled} ({config.lateness_probability:.0%} probability)")
print(f"  Manifest:      {config.manifest_enabled}")
print(f"  Done flag:     {config.done_flag_enabled}")

## Step 3 — Run the Simulation

**What we're about to do:** Execute the file-drop simulation. The simulator
will partition the source data into daily batches, write Parquet files to
date-partitioned folders, generate manifests and done-flags, and randomly
delay some drops to simulate late arrivals.

**Why this matters:** This creates the exact folder structure your Fabric
Data Pipeline would encounter in production. You can point your pipeline
at this output and test its behavior end-to-end — including how it handles
the 15% of drops that arrive late.

**What to expect:** A result showing all files written, any late arrivals,
and the folder structure.

In [ ]:
sim = FileDropSimulator(result.tables, config)
drop_result = sim.run()

print(f"Simulation Complete")
print(f"=" * 50)
print(f"  Total files written:  {len(drop_result.files_written):,}")
print(f"  Total batches:        {drop_result.batch_count}")
print(f"  Late arrivals:        {drop_result.late_count}")
print(f"  On-time arrivals:     {drop_result.batch_count - drop_result.late_count}")

# Batch arrival report
print(f"\n=== Batch Arrival Report ===")
print(f"{'Date':<14} {'Status':<10} {'Files':>6} {'Rows':>8} {'Delay':>10}")
print("-" * 52)

for batch in drop_result.batches:
    status = "LATE" if batch.is_late else "on-time"
    delay = batch.delay if batch.is_late else "-"
    print(f"  {batch.date:<12} {status:<10} {batch.file_count:>6} {batch.row_count:>8,} {str(delay):>10}")

late_batches = [b for b in drop_result.batches if b.is_late]
print(f"\n{len(late_batches)} of {len(drop_result.batches)} batches arrived late ({len(late_batches)/len(drop_result.batches)*100:.0f}%)")

## Step 4 — Chaos Injection and Validation

**What we're about to do:** First, apply chaos injection to selected file drops —
simulating corrupted files, missing columns, and truncated data. Then run
Spindle's built-in validation gates to verify that quality checks catch the
injected problems.

**Why this matters:** Production pipelines break in subtle ways: a column
gets renamed upstream, a Parquet file is truncated mid-write, a schema
evolves without warning. Chaos injection lets you test your pipeline's
error handling, while validation gates confirm that your quality checks
actually catch these problems. Files with chaos applied should trigger
validation failures — if they don't, your quality gates have gaps.

**What to expect:** A chaos report showing affected files, followed by a
validation report showing which batches pass and which fail.

In [ ]:
from sqllocks_spindle.simulation.chaos import ChaosConfig

chaos_config = ChaosConfig(
    corruption_probability=0.1,     # 10% of files get corrupted
    missing_column_probability=0.05, # 5% chance of a dropped column
    truncation_probability=0.05,     # 5% chance of truncated data
    seed=42,
)

chaos_result = sim.apply_chaos(chaos_config)

print("=== Chaos Injection Report ===")
print(f"  Files affected:  {chaos_result.affected_count}")
print(f"  Files untouched: {len(drop_result.files_written) - chaos_result.affected_count}")

print(f"\n=== Affected Files ===")
for item in chaos_result.items:
    print(f"  {item.file_path}")
    print(f"    Type: {item.chaos_type}  |  Detail: {item.detail}")

# Now validate — chaos-injected files should trigger failures
validation = sim.validate()

print(f"\n=== Validation Gate Results ===")
print(f"{'Date':<14} {'Status':<10} {'Checks':>7} {'Passed':>7} {'Failed':>7}")
print("-" * 50)

pass_count = 0
fail_count = 0
for v in validation.results:
    status = "PASS" if v.passed else "FAIL"
    print(f"  {v.date:<12} {status:<10} {v.total_checks:>7} {v.passed_checks:>7} {v.failed_checks:>7}")
    if v.passed:
        pass_count += 1
    else:
        fail_count += 1

print(f"\nSummary: {pass_count} passed, {fail_count} failed")

if fail_count > 0:
    print(f"\n=== Failure Details ===")
    for v in validation.results:
        if not v.passed:
            for err in v.errors:
                print(f"  [{v.date}] {err}")

## Step 5 — Review the Complete Structure

**What we're about to do:** Display the full directory structure that was
generated — showing the date-partitioned folders, data files, manifests,
and done-flags.

**Why this matters:** This is the exact structure you would upload to your
Fabric Lakehouse `Files/` folder or ADLS landing zone. Your Data Pipeline
can be configured to watch this folder structure and process each daily
batch as it appears.

**What to expect:** A tree view of the output directory.

In [ ]:
import os

base = config.base_path
print(f"=== File-Drop Directory Structure ===")
print(f"{base}/")

total_size = 0
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * (level + 1)
    folder = os.path.basename(root)
    if level > 0:
        print(f"{indent}{folder}/")
    sub_indent = "  " * (level + 2)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_size += size
        print(f"{sub_indent}{file:<35} {size:>10,} bytes")

print(f"\nTotal disk usage: {total_size:,.0f} bytes ({total_size/1024:.1f} KB)")
print(f"\nThis structure can be uploaded directly to a Fabric Lakehouse Files/ folder")
print(f"or an ADLS Gen2 landing zone for end-to-end pipeline testing.")

## What's Next?

You've built a complete file-drop test harness — realistic daily drops with
late arrivals, chaos injection, and validation gates. This is everything you
need to test a Fabric ingestion pipeline end-to-end.

| Notebook | What You'll Learn |
|----------|-------------------|
| **F05: Chaos Pipeline** | Deep dive into chaos injection patterns |
| **F01: Medallion Architecture** | Build bronze/silver/gold layers from file drops |
| **T14: Day 2 — Incremental** | Generate CDC-style deltas for merge/upsert testing |
| **F09: Cross-Domain Enterprise** | Scale up to multi-domain enterprise datasets |

**Key takeaways:**
- `FileDropSimulator` creates date-partitioned file drops with manifests and done-flags
- Late arrival simulation tests your pipeline's handling of out-of-order data
- Chaos injection (corruption, missing columns, truncation) validates error handling
- Validation gates verify manifest accuracy and schema consistency
- The output structure maps directly to Fabric Lakehouse and ADLS landing zones